# Satellite Image Classification

In [1]:
import os
import random
import kagglehub
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

## Reproducibility & config

In [2]:
@dataclass
class Config:
    data_dir: str = os.path.join(
        kagglehub.dataset_download("mahmoudreda55/satellite-image-classification"),
        "data"
    )

    image_size: int = 64
    batch_size: int = 32
    num_workers: int = 2
    val_split: float = 0.2
    seed: int = 42

    epochs: int = 10
    lr: float = 1e-3
    weight_decay: float = 1e-4


cfg = Config()

print("Path to dataset files:", cfg.data_dir)


def seed_everything(seed: int) -> None:
    """Set random seeds."""
    random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(False)


seed_everything(cfg.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

100%|██████████| 21.8M/21.8M [00:02<00:00, 11.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mahmoudreda55/satellite-image-classification/versions/1/data
Using device: cuda


## Data

In [3]:
def build_transforms(image_size: int) -> Tuple[transforms.Compose, transforms.Compose]:
    """Return train/val transforms."""
    train_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
    ])
    val_tfms = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])
    return train_tfms, val_tfms

train_tfms, val_tfms = build_transforms(cfg.image_size)

# Load once, then wrap with different transforms using a small helper.
base_ds = datasets.ImageFolder(cfg.data_dir)

class TransformDataset(torch.utils.data.Dataset):
    """Apply a transform to an existing dataset."""
    def __init__(self, ds, tfm):
        self.ds = ds
        self.tfm = tfm

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        x, y = self.ds[idx]
        return self.tfm(x), y

# Split indices
val_len = int(len(base_ds) * cfg.val_split)
train_len = len(base_ds) - val_len
train_ds_raw, val_ds_raw = random_split(base_ds, [train_len, val_len], generator=torch.Generator().manual_seed(cfg.seed))

train_ds = TransformDataset(train_ds_raw, train_tfms)
val_ds = TransformDataset(val_ds_raw, val_tfms)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

class_names: List[str] = base_ds.classes
num_classes = len(class_names)

print(f"Samples: train={len(train_ds)} val={len(val_ds)} classes={class_names}")

Samples: train=4505 val=1126 classes=['cloudy', 'desert', 'green_area', 'water']


## Model

In [4]:
def conv_block(in_ch: int, out_ch: int, pool: bool = False) -> nn.Sequential:
    """Conv-BN-ReLU (+ optional MaxPool)."""
    layers: List[nn.Module] = [
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    ]
    if pool:
        layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)

class ResNet9(nn.Module):
    """Small residual CNN for 64x64 images."""
    def __init__(self, in_ch: int, num_classes: int):
        super().__init__()
        self.conv1 = conv_block(in_ch, 64)
        self.conv2 = conv_block(64, 128, pool=True)
        self.res1 = nn.Sequential(conv_block(128, 128), conv_block(128, 128))
        self.conv3 = conv_block(128, 256, pool=True)
        self.conv4 = conv_block(256, 512, pool=True)
        self.res2 = nn.Sequential(conv_block(512, 512), conv_block(512, 512))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.res1(x) + x
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.res2(x) + x
        return self.head(x)

model = ResNet9(in_ch=3, num_classes=num_classes).to(device)
print(model)

ResNet9(
  (conv1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (conv2): Sequential(
    (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (res1): Sequential(
    (0): Sequential(
      (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (1): Sequential(
      (0): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, trac

## Metrics & helpers

In [5]:
@torch.no_grad()
def accuracy_from_logits(logits: torch.Tensor, targets: torch.Tensor) -> float:
    """Batch accuracy."""
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> Dict[str, float]:
    """Validation loss and accuracy."""
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb, reduction="sum")
        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        total_correct += (preds == yb).sum().item()
        total_seen += yb.numel()

    return {
        "val_loss": total_loss / max(total_seen, 1),
        "val_acc": total_correct / max(total_seen, 1),
    }

def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> Dict[str, float]:
    """Train for one epoch."""
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_seen += yb.numel()

    return {
        "train_loss": total_loss / max(total_seen, 1),
        "train_acc": total_correct / max(total_seen, 1),
    }

## Train


In [6]:
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

history: List[Dict[str, float]] = []

for epoch in range(1, cfg.epochs + 1):
    train_metrics = train_one_epoch(model, train_loader, optimizer)
    val_metrics = evaluate(model, val_loader)

    row = {"epoch": epoch, **train_metrics, **val_metrics}
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{cfg.epochs} | "
        f"train_loss={row['train_loss']:.4f} train_acc={row['train_acc']:.4f} | "
        f"val_loss={row['val_loss']:.4f} val_acc={row['val_acc']:.4f}"
    )

Epoch 01/10 | train_loss=0.3978 train_acc=0.8435 | val_loss=0.7129 val_acc=0.7513
Epoch 02/10 | train_loss=0.2889 train_acc=0.8926 | val_loss=1.5147 val_acc=0.7584
Epoch 03/10 | train_loss=0.2308 train_acc=0.9185 | val_loss=0.3438 val_acc=0.8908
Epoch 04/10 | train_loss=0.1837 train_acc=0.9358 | val_loss=0.7339 val_acc=0.7469
Epoch 05/10 | train_loss=0.1850 train_acc=0.9365 | val_loss=0.1010 val_acc=0.9707
Epoch 06/10 | train_loss=0.2113 train_acc=0.9305 | val_loss=0.2733 val_acc=0.8845
Epoch 07/10 | train_loss=0.1808 train_acc=0.9421 | val_loss=0.1225 val_acc=0.9609
Epoch 08/10 | train_loss=0.1707 train_acc=0.9401 | val_loss=0.2406 val_acc=0.8943
Epoch 09/10 | train_loss=0.1564 train_acc=0.9465 | val_loss=1.0332 val_acc=0.7940
Epoch 10/10 | train_loss=0.1764 train_acc=0.9356 | val_loss=0.1170 val_acc=0.9654


## Save model

In [7]:
out_dir = Path("model")
out_dir.mkdir(exist_ok=True)

model_path = out_dir / "model.pth"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "classes": class_names,
        "image_size": cfg.image_size,
    },
    model_path,
)

print(f"Saved: {model_path.resolve()}")


Saved: /content/model/model.pth
